In [1]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
!pip install --upgrade -qqq uv
try: import numpy, PIL; get_numpy = f"numpy=={numpy.__version__}"; get_pil = f"pillow=={PIL.__version__}"
except: get_numpy = "numpy"; get_pil = "pillow"
try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
except: is_t4 = False
get_vllm, get_triton = ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm==0.10.2", "triton")
!uv pip install -qqq --upgrade     unsloth {get_vllm} {get_numpy} {get_pil} torchvision bitsandbytes xformers
!uv pip install -qqq {get_triton}
!uv pip install "huggingface_hub>=0.34.0" "datasets==4.3.0
!uv pip install transformers==4.57.3
!uv pip install --no-deps trl==0.22.2

In [2]:
import os
import torch
import pandas as pd
from datasets import Dataset

from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator

from trl import SFTTrainer, SFTConfig


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-11-30 10:29:11.635200: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764498551.806125     181 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764498551.851508     181 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


INFO 11-30 10:29:36 [__init__.py:244] Automatically detected platform cuda.
ERROR 11-30 10:29:38 [fa_utils.py:57] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
!pip install --upgrade transformers

In [3]:
model_name = "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit"

max_seq_length = 16384  # or 4096 / 8192 depending on your GPU
dtype = torch.bfloat16

model, tokenizer = FastVisionModel.from_pretrained(
    model_name          = model_name,
    max_seq_length      = max_seq_length,
    load_in_4bit        = True,
    torch_dtype         = dtype,
    use_gradient_checkpointing = "unsloth",
)


==((====))==  Unsloth 2025.11.4: Fast Qwen3_Vl patching. Transformers: 4.57.3. vLLM: 0.9.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.72G [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

In [4]:
model = FastVisionModel.get_peft_model(
    model,
    # In the video they leave vision layers frozen
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,

    r           = 16,
    lora_alpha  = 16,
    lora_dropout= 0.0,
    bias        = "none",
    random_state= 3407,
    use_gradient_checkpointing = "unsloth"
)

try:
  model.print_trainable_parameters()
except Exception as e:
  trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
  total = sum(p.numel() for p in model.parameters())
  print(f"Trainable: {trainable} | Total: {total} | {trainable/total*100:.2f}%")


trainable params: 43,646,976 || all params: 8,810,770,672 || trainable%: 0.4954


In [ ]:
!unzip /content/TrarinImage.zip -d /content/Image

In [19]:
from datasets import Dataset
from PIL import Image
import pandas as pd
import os

# Paths from your description
CSV_PATH = "/kaggle/input/poli-meme-decode-cuet-cse-fest/PoliMemeDecode/Train/Train.csv"
IMG_DIR = "/kaggle/input/poli-meme-decode-cuet-cse-fest/PoliMemeDecode/Train/Image"

# 1. Load first 500 rows
train_df = pd.read_csv(CSV_PATH).head(2800)

TEXT_CONTENT = """You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:
Political: A meme depicting a political figure, making fun of a political figure, making fun of a political ideology or political party or mentioning a political figure.
NonPolitical: A meme that is non political, contains humor or sarcasm regarding any non political issues or memes about relatable things or day to day life.
Output a single word without providing any reasoning or explanations"""

# 2. Create a HF Dataset with raw columns first
#    We keep Image_name and Label, we'll add decoded_image
raw_ds = Dataset.from_pandas(train_df, preserve_index=False)

# 3. Load, resize to 512x512, convert to RGB -> decoded_image
def load_and_resize(example):
    img_path = os.path.join(IMG_DIR, example["Image_name"])
    img = Image.open(img_path)
    img = img.resize((512, 512))
    if img.mode != "RGB":
        img = img.convert("RGB")
    example["decoded_image"] = img
    return example

dataset = raw_ds.map(load_and_resize)

# 4. Build MathVista-style prompt structure: list with image placeholder + text
def make_conversation(example):
    prompt = [
        {
            "role": "user",
            "content": [
                {"type": "image"},  # placeholder; actual image kept in decoded_image
                {"type": "text", "text": TEXT_CONTENT},
            ],
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": example["Label"]},
            ],
        },
    ]
    return {
        "prompt": prompt,
        "image": example["decoded_image"],  # keep image separate (like MathVista)
        "answer": example["Label"],
    }

dataset = dataset.map(make_conversation)

# 5. Remove old image column (if any) and rename decoded_image -> image
#    In our case, we already use 'image' key above, so we can just drop decoded_image.
if "decoded_image" in dataset.column_names:
    dataset = dataset.remove_columns("decoded_image")

# 6. Apply chat template to build final text prompt string
#    tokenizer must already be created (from your FastVisionModel)
def apply_template(example):
    example["prompt"] = tokenizer.apply_chat_template(
        example["prompt"],
        tokenize=False,
        add_generation_prompt=True,  # include assistant tag
    )
    return example

dataset = dataset.map(apply_template)

print(dataset)
print("Columns:", dataset.column_names)
print("Sample prompt:", dataset[0]["prompt"][:400])
print("Image type:", type(dataset[0]["image"]))


Map:   0%|          | 0/2800 [00:00<?, ? examples/s]

Map:   0%|          | 0/2800 [00:00<?, ? examples/s]

Map:   0%|          | 0/2800 [00:00<?, ? examples/s]

Dataset({
    features: ['Image_name', 'Label', 'prompt', 'image', 'answer'],
    num_rows: 2800
})
Columns: ['Image_name', 'Label', 'prompt', 'image', 'answer']
Sample prompt: <|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:
Political: A meme depicting a political figure, making fun of a political figure, making fun of a political ideology or political party or mentioning a political figure.
NonPolitical: A meme that is non pol
Image type: <class 'PIL.PngImagePlugin.PngImageFile'>


In [21]:
import re

def meme_correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    """
    prompts: list[str]       – full prompts
    completions: list[str]   – model outputs
    answer: list[str]        – ground truth labels, e.g. ["Political", "NonPolitical", ...]
    """

    # Log the first example like the original function
    q = prompts[0]
    print(
        "-" * 20,
        f"Question:\n{q}",
        f"\nAnswer:\n{answer[0]}",
        f"\nResponse:{completions[0]}",
    )

    def get_label(text: str) -> str:
        # Clean up completion
        text = text.strip()
        text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
        text = re.sub(r"<.*?>", "", text).strip()
        pred = text.lower()

        if "nonpolitical" in pred or "non-political" in pred or "non political" in pred:
            return "nonpolitical"
        if "political" in pred:
            return "political"
        return ""

    def gold_label(a: str) -> str:
        g = a.lower().strip()
        return "nonpolitical" if "non" in g else "political"

    # Single list comprehension, like the original function
    return [
        2.0 if get_label(c) == gold_label(a) else -2.0
        for c, a in zip(completions, answer)
    ]


In [13]:
dataset[100]["prompt"]

'<|im_start|>user\n<|vision_start|><|image_pad|><|vision_end|>You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:\nPolitical: A meme depicting a political figure, making fun of a political figure, making fun of a political ideology or political party or mentioning a political figure.\nNonPolitical: A meme that is non political, contains humor or sarcasm regarding any non political issues or memes about relatable things or day to day life.\nOutput a single word without providing any reasoning or explanations<|im_end|>\n<|im_start|>assistant\nNonPolitical<|im_end|>\n<|im_start|>assistant\n'

In [14]:
image = dataset[100]["image"]
prompt = dataset[100]["prompt"]

inputs = tokenizer(
    image,
    prompt,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 1024,
                   use_cache = True, temperature = 1.0, min_p = 0.1)

NonPolitical<|im_end|>


In [22]:
from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 1,
    log_completions = False,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4, # Increase to 4 for smoother training
    num_generations = 2, # Decrease if out of memory
    max_prompt_length = 1024,
    max_completion_length = 1024,
    num_train_epochs = 0.5, # Set to 1 for a full training run
    # max_steps = 60,
    save_steps = 60,
    max_grad_norm = 0.1,
    report_to = "none", # Can use Weights & Biases
    output_dir = "outputs",

    # Below enables GSPO:
    importance_sampling_level = "sequence",
    mask_truncated_completions = False,
    loss_type = "dr_grpo",
)

In [ ]:
trainer = GRPOTrainer(
    model = model,
    args = training_args,
    # Pass the processor to handle multimodal inputs
    processing_class = tokenizer,
    reward_funcs = [
      meme_correctness_reward_func
    ],
    train_dataset = dataset,
)

trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,800 | Num Epochs = 1 | Total steps = 700
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 43,646,976 of 8,810,770,672 (0.50% trained)


-------------------- Question:
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:
Political: A meme depicting a political figure, making fun of a political figure, making fun of a political ideology or political party or mentioning a political figure.
NonPolitical: A meme that is non political, contains humor or sarcasm regarding any non political issues or memes about relatable things or day to day life.
Output a single word without providing any reasoning or explanations<|im_end|>
<|im_start|>assistant
Political<|im_end|>
<|im_start|>assistant
 
Answer:
Political 
Response:Political


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / meme_correctness_reward_func / mean,rewards / meme_correctness_reward_func / std
1,0.000000,2.000000,0.000000,2.000000,2.000000,2.000000,0.000000,2.000000,2.000000,2.000000,0.000618,2.000000,0.000000
2,0.000000,2.000000,0.000000,2.500000,2.000000,3.000000,0.000000,2.500000,2.000000,3.000000,0.000003,2.000000,0.000000
3,0.000000,2.000000,0.000000,2.000000,2.000000,2.000000,0.000000,2.000000,2.000000,2.000000,0.089882,2.000000,0.000000
4,0.000000,2.000000,0.000000,3.000000,3.000000,3.000000,0.000000,3.000000,3.000000,3.000000,0.000001,2.000000,0.000000
5,0.000000,2.000000,0.000000,2.500000,2.000000,3.000000,0.000000,2.500000,2.000000,3.000000,0.000001,2.000000,0.000000
6,0.000000,2.000000,0.000000,2.500000,2.000000,3.000000,0.000000,2.500000,2.000000,3.000000,0.068689,2.000000,0.000000
7,0.000000,2.000000,0.000000,3.000000,3.000000,3.000000,0.000000,3.000000,3.000000,3.000000,0.000002,2.000000,0.000000
8,0.000000,2.000000,0.000000,3.000000,3.000000,3.000000,0.000000,3.000000,3.000000,3.000000,0.000000,2.000000,0.000000
9,0.000000,2.000000,0.000000,2.000000,2.000000,2.000000,0.000000,2.000000,2.000000,2.000000,0.064525,2.000000,0.000000
10,0.000000,2.000000,0.000000,3.000000,3.000000,3.000000,0.000000,3.000000,3.000000,3.000000,0.000003,2.000000,0.000000


-------------------- Question:
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:
Political: A meme depicting a political figure, making fun of a political figure, making fun of a political ideology or political party or mentioning a political figure.
NonPolitical: A meme that is non political, contains humor or sarcasm regarding any non political issues or memes about relatable things or day to day life.
Output a single word without providing any reasoning or explanations<|im_end|>
<|im_start|>assistant
NonPolitical<|im_end|>
<|im_start|>assistant
 
Answer:
NonPolitical 
Response:NonPolitical
-------------------- Question:
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:
Political: A meme depicting a p

In [25]:
trainer.train(resume_from_checkpoint="outputs/checkpoint-360")


The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,800 | Num Epochs = 1 | Total steps = 700
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 43,646,976 of 8,810,770,672 (0.50% trained)


-------------------- Question:
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:
Political: A meme depicting a political figure, making fun of a political figure, making fun of a political ideology or political party or mentioning a political figure.
NonPolitical: A meme that is non political, contains humor or sarcasm regarding any non political issues or memes about relatable things or day to day life.
Output a single word without providing any reasoning or explanations<|im_end|>
<|im_start|>assistant
NonPolitical<|im_end|>
<|im_start|>assistant
 
Answer:
NonPolitical 
Response:NonPolitical


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / meme_correctness_reward_func / mean,rewards / meme_correctness_reward_func / std
361,0.000000,2.000000,0.000000,3.000000,3.000000,3.000000,0.000000,3.000000,3.000000,3.000000,0.000004,2.000000,0.000000
362,0.000000,2.000000,0.000000,3.000000,3.000000,3.000000,0.000000,3.000000,3.000000,3.000000,0.000000,2.000000,0.000000
363,0.000000,2.000000,0.000000,3.000000,3.000000,3.000000,0.000000,3.000000,3.000000,3.000000,0.000000,2.000000,0.000000
364,0.000000,2.000000,0.000000,3.000000,3.000000,3.000000,0.000000,3.000000,3.000000,3.000000,0.000000,2.000000,0.000000
365,0.000000,2.000000,0.000000,2.500000,2.000000,3.000000,0.000000,2.500000,2.000000,3.000000,0.082239,2.000000,0.000000
366,0.000000,2.000000,0.000000,3.000000,3.000000,3.000000,0.000000,3.000000,3.000000,3.000000,0.000002,2.000000,0.000000
367,0.000000,2.000000,0.000000,3.000000,3.000000,3.000000,0.000000,3.000000,3.000000,3.000000,0.000000,2.000000,0.000000
368,0.000000,2.000000,0.000000,2.500000,2.000000,3.000000,0.000000,2.500000,2.000000,3.000000,0.000011,2.000000,0.000000
369,0.000000,2.000000,0.000000,3.000000,3.000000,3.000000,0.000000,3.000000,3.000000,3.000000,0.000001,2.000000,0.000000
370,0.000000,2.000000,0.000000,2.500000,2.000000,3.000000,0.000000,2.500000,2.000000,3.000000,0.000001,2.000000,0.000000


-------------------- Question:
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:
Political: A meme depicting a political figure, making fun of a political figure, making fun of a political ideology or political party or mentioning a political figure.
NonPolitical: A meme that is non political, contains humor or sarcasm regarding any non political issues or memes about relatable things or day to day life.
Output a single word without providing any reasoning or explanations<|im_end|>
<|im_start|>assistant
NonPolitical<|im_end|>
<|im_start|>assistant
 
Answer:
NonPolitical 
Response:NonPolitical
-------------------- Question:
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:
Political: A meme depicting a p

TrainOutput(global_step=700, training_loss=2.490450814778766e-06, metrics={'train_runtime': 4096.2908, 'train_samples_per_second': 0.342, 'train_steps_per_second': 0.171, 'total_flos': 0.0, 'train_loss': 2.490450814778766e-06})

In [26]:
model.push_to_hub_merged("arafid/Q38B-all-images-fully-trained", tokenizer, save_method = "merged_16bit", token = "")

Unsloth: Saving to arafid/Q38B-all-images-fully-trained will fail, but using a temp folder works! Switching to a temp folder then uploading!


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Unsloth: Merging weights into 16bit:  75%|███████▌  | 3/4 [05:00<01:39, 99.51s/it] 

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [05:44<00:00, 86.05s/it]


Unsloth: Merge process complete. Saved to `/tmp/tmp52fjyqnx`


## STFT training test

In [ ]:
from transformers import DefaultDataCollator
from trl import SFTTrainer, SFTConfig
from unsloth import FastVisionModel
import torch

# ===== 1. Tokenize prompt + image =====
def preprocess(example):
    out = tokenizer(
        text   = example["prompt"],
        images = example["image"],
        return_tensors = None,
        padding = False,
    )
    # Flatten if processor returns [[...]]
    if isinstance(out["input_ids"][0], list):
        out["input_ids"]      = out["input_ids"][0]
        out["attention_mask"] = out["attention_mask"][0]
    return out

tokenized_dataset = dataset.map(
    preprocess,
    remove_columns = dataset.column_names,  # drop Image_name, Label, prompt, image, answer
)

print("Tokenized columns:", tokenized_dataset.column_names)
print("Sample input_ids length:", len(tokenized_dataset[0]["input_ids"]))

# ===== 2. SFT Config =====
FastVisionModel.for_training(model)

training_args = SFTConfig(
    # Optimization
    learning_rate              = 5e-6,
    adam_beta1                 = 0.9,
    adam_beta2                 = 0.99,
    weight_decay               = 0.1,
    warmup_ratio               = 0.1,
    lr_scheduler_type          = "cosine",
    optim                      = "adamw_8bit",
    max_grad_norm              = 0.1,
    
    # Training loop
    per_device_train_batch_size= 1,
    gradient_accumulation_steps= 4,      # effective batch 4
    num_train_epochs           = 1.0,
    # max_steps                = 60,     # or use steps
    
    # Logging and saving
    logging_steps              = 1,
    save_steps                 = 60,
    output_dir                 = "outputs_sft",
    report_to                  = "none",
    
    # Required for tokenized datasets
    remove_unused_columns      = False,
    
    # Precision
    fp16                       = not torch.cuda.is_bf16_supported(),
    bf16                       = torch.cuda.is_bf16_supported(),
    
    seed                       = 3407,
    dataloader_num_workers     = 0,
)

# ===== 3. SFTTrainer with DefaultDataCollator =====
trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    data_collator = DefaultDataCollator(),   # simple collator for tokenized data
    args          = training_args,
    train_dataset = tokenized_dataset,       # already tokenized
)

train_results = trainer.train()
print(train_results)


In [ ]:
"""You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:
Political: A meme depicting a political figure, making fun of a political figure, making fun of a political ideology or political party or mentioning a political figure.If a meme makes fun of or refers to a political party then its a political meme(Political party includes but is not limited to 'Jamat','Jamat islami party','NCP or national citizen party','BAL-Bangladesh Awami League','BNP-Bangladesh National Party','Republican','Democrats' etc).
NonPolitical: A meme that is non political, contains humor or sarcasm regarding any non political issues or memes about relatable things or day to day life.
Output a single word without providing any reasoning or explanations"""

In [28]:
inputs = tokenizer(
    image,
    prompt,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 1024,
                   use_cache = True, temperature = 1.0, min_p = 0.1)

#output
#NonPolitical<|im_end|>
prmopt = """You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:
Political: A meme depicting a political figure, making fun of a political figure, making fun of a political ideology or political party or mentioning a political figure.If a meme makes fun of or refers to a political party then its a political meme(Political party includes but is not limited to 'Jamat','Jamat islami party','NCP or national citizen party','BAL-Bangladesh Awami League','BNP-Bangladesh National Party','Republican','Democrats' etc).
NonPolitical: A meme that is non political, contains humor or sarcasm regarding any non political issues or memes about relatable things or day to day life.
Output a single word without providing any reasoning or explanations"""

test_image_folder = '/kaggle/input/poli-meme-decode-cuet-cse-fest/PoliMemeDecode/Test/Image'
test_csv = '/kaggle/input/poli-meme-decode-cuet-cse-fest/PoliMemeDecode/Test/Test.csv'

NonPolitical<|im_end|>
